In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/crisis-text-triage")

In [ ]:
folders = [
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "models",
    PROJECT_ROOT / "reports",
]

In [ ]:
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:
PROJECT_ROOT

In [ ]:
for path in sorted(PROJECT_ROOT.rglob("*")):
    print(path.relative_to(PROJECT_ROOT))

In [ ]:
!pip install -q datasets

In [ ]:
from datasets  import load_dataset

In [ ]:
DATASET_NAME = "QCRI/HumAID-all"

In [ ]:
dataset = load_dataset(
    DATASET_NAME,
    verification_mode="no_checks"
)

In [ ]:
dataset

In [ ]:
type(dataset)

In [ ]:
dataset["train"]

In [ ]:
train_data = dataset["train"]

In [ ]:
train_data[0]

In [ ]:
for index in range(5):
  example = train_data[index]
  print(f"{index+ 1}, text:{ example["tweet_text"]}, label: {example["class_label"]}")

In [ ]:
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

for split_name, split_data in dataset.items():
    output_path = RAW_DATA_DIR / f"humaid_{split_name}.parquet"

    split_data.to_parquet(str(output_path))

    print(f"{split_name}: {output_path}")

In [ ]:
for file_path in sorted(RAW_DATA_DIR.glob("*.parquet")):
    size_mb = file_path.stat().st_size / (1024 ** 2)
    print(f"{file_path.name:<30} {size_mb:.2f} MB")

In [ ]:
type(train_data)

In [ ]:
train_df = dataset["train"].to_pandas()
train_df.head()

In [ ]:
train_df.nunique()

In [ ]:
train_df.shape

In [ ]:
train_df.isna().sum()

In [ ]:
train_df.duplicated().sum()

In [ ]:
train_df["class_label"].value_counts(normalize=True)

In [ ]:
imbalance_ratio = ((train_df["class_label"].value_counts(normalize=True)).max()) / ((train_df["class_label"].value_counts(normalize=True)).min())
imbalance_ratio

In [ ]:
audit_df = train_df.copy()

In [ ]:
audit_df

In [ ]:
audit_df["char_count"] = audit_df["tweet_text"].str.len()

In [ ]:
audit_df["word_count"] = audit_df["tweet_text"].str.split().str.len()

In [ ]:
audit_df

In [ ]:
audit_df["char_count"].describe()

In [ ]:
audit_df["word_count"].describe()

In [ ]:
audit_df["tweet_text"]

In [ ]:
audit_df["is_retweet"] = audit_df["tweet_text"].str.startswith("RT ")
audit_df["has_url"] = audit_df["tweet_text"].str.contains(
    r"http|www",
    case=False,
    na=False
)
audit_df["has_mention"] = audit_df["tweet_text"].str.contains("@", na=False)
audit_df["has_hashtag"] = audit_df["tweet_text"].str.contains("#", na=False)
audit_df["has_exclamation"] = audit_df["tweet_text"].str.contains("!", na=False)

In [ ]:
audit_df.head()

In [ ]:
audit_df.nlargest(5, "char_count")[["tweet_text", "class_label", "char_count", "word_count"]]

In [ ]:
audit_df.nsmallest(5, "char_count")[["tweet_text", "class_label", "char_count", "word_count"]]

In [ ]:
signal_columns = [
    "is_retweet",
    "has_url",
    "has_mention",
    "has_hashtag",
    "has_exclamation"
]

In [ ]:
signal_summary = (audit_df[signal_columns].mean().mul(100).round(2))
signal_summary

In [ ]:
audit_df["mention_count"] = audit_df["tweet_text"].str.count(r"@\w+")

In [ ]:
audit_df.nlargest(5, "char_count")[["mention_count","tweet_text", "class_label", "char_count", "word_count"]]

In [ ]:
audit_df["mention_count"].describe()

In [ ]:
audit_df["mention_count"].value_counts().sort_index().head(10)

In [ ]:
audit_df.nlargest(10, "mention_count")[["tweet_text", "class_label", "mention_count", "char_count"]]

In [ ]:
audit_df["has_mojibake"] = audit_df['tweet_text'].str.contains(r"[ðÃâ]")

In [ ]:
audit_df[audit_df["has_mojibake"]][
    ["tweet_text", "class_label"]
].head(10)

In [ ]:
for text in audit_df.nsmallest(10, "char_count")["tweet_text"]:
    print(repr(text))

In [ ]:
import unicodedata

In [ ]:
suspicious_text = audit_df.nsmallest(1, "char_count")["tweet_text"].iloc[0]

In [ ]:
shortest_texts = audit_df.nsmallest(10, "char_count")["tweet_text"]

In [ ]:
for text in shortest_texts:
    print(f"\nTEXT: {text!r}")

    for character in text:
        if ord(character) > 127:
            print(
                f"Character: {character!r} | "
                f"Code: {hex(ord(character))} | "
                f"Name: {unicodedata.name(character, 'UNKNOWN')}"
            )

In [ ]:
greek_extended_pattern = r"[\u1F00-\u1FFF]"

audit_df["has_greek_extended"] = audit_df["tweet_text"].str.contains(
    greek_extended_pattern,
    regex=True,
    na=False
)

greek_extended_count = audit_df["has_greek_extended"].sum()

greek_extended_percentage = (
    audit_df["has_greek_extended"].mean() * 100
)

print(f"Message count: {greek_extended_count}")
print(f"Percentage: {greek_extended_percentage:.2f}%")

#preprocessing

In [ ]:
import re
import unicodedata

In [ ]:
def normalize_basic_text(text: str, lowercase: bool = True) -> str:
    normalized_text = unicodedata.normalize("NFKC", text)

    if lowercase:
        normalized_text = normalized_text.lower()

    normalized_text = re.sub(r"\s+", " ", normalized_text)
    normalized_text = normalized_text.strip()

    return normalized_text

In [ ]:
test_text = "  HELP!!!\n\nWe need   WATER  "

In [ ]:
print(normalize_basic_text(test_text))
print(normalize_basic_text(test_text, lowercase=False))

In [ ]:
def replace_mentions(text: str, replacement: str = "<user>") -> str:
    result = re.sub(r"@\w+", "<user>", text)
    return result

In [ ]:
test_messages = [
    "RT @RachelAndJun: Please help @UNICEF",
    "@CNN pls send help",
    "No mention here",
]

for text in test_messages:
    print(replace_mentions(text))

In [ ]:
def replace_urls(text: str, replacement: str = "<url>") -> str:
    result = re.sub(
        r"https?://\S+|www\.\S+",
        replacement,
        text
    )

    return result

In [ ]:
test_messages = [
    "Donate at https://example.com/help",
    "More info: http://news.com/article",
    "Visit www.example.org now",
    "There is no link here",
]

for text in test_messages:
    print(replace_urls(text))

In [ ]:
def normalize_hashtags(text: str) -> str:
    result = re.sub(
        r"#(\w+)",
        r"\1",
        text
    )

    return result

In [ ]:
test_messages = [
    "Pray for everyone affected by #EcuadorEarthquake",
    "#FloodHelp is urgently needed",
    "No hashtag here",
]

for text in test_messages:
    print(normalize_hashtags(text))

In [ ]:
def preprocess_text(text: str) -> str:
  text = normalize_basic_text(text)
  text = replace_urls(text)
  text = replace_mentions(text)
  text = normalize_hashtags(text)
  return text

In [ ]:
test_text = """
RT @ReliefTeam: HELP needed at #EcuadorEarthquake!
Donate: https://example.com/help
"""

preprocess_text(test_text)

In [ ]:
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()
validation_df = dataset["validation"].to_pandas()

In [ ]:
train_df["text_minimal"] = train_df["tweet_text"].apply(preprocess_text)
test_df["text_minimal"] = test_df["tweet_text"].apply(preprocess_text)
validation_df["text_minimal"] = validation_df["tweet_text"].apply(preprocess_text)

In [ ]:
train_df["text_minimal"].isna().sum(), train_df["text_minimal"].str.strip().eq("").sum(),

In [ ]:
train_df[["tweet_text", "text_minimal"]].head()

In [ ]:
changed_mask = train_df["tweet_text"].ne(train_df["text_minimal"])

changed_count = changed_mask.sum()
missing_count = train_df["text_minimal"].isna().sum()
empty_count = train_df["text_minimal"].str.strip().eq("").sum()

print("Changed:", changed_count)
print("Missing:", missing_count)
print("Empty:", empty_count)

In [ ]:
changed_mask = train_df["tweet_text"].ne(train_df["text_minimal"])

changed_examples = (
    train_df.loc[
        changed_mask,
        ["tweet_text", "text_minimal", "class_label"]
    ]
    .sample(n=5, random_state=42)
)

unchanged_examples = (
    train_df.loc[
        ~changed_mask,
        ["tweet_text", "text_minimal", "class_label"]
    ]
    .sample(n=5, random_state=42)
)

print("CHANGED EXAMPLES")
display(changed_examples)

print("\nUNCHANGED EXAMPLES")
display(unchanged_examples)

In [ ]:
train_df

In [ ]:
X_train = train_df["text_minimal"]
y_train = train_df["class_label"]

X_val = validation_df["text_minimal"]
y_val = validation_df["class_label"]

X_test = test_df["text_minimal"]
y_test = test_df['class_label']

In [ ]:
train_size = X_train.shape
validation_size = X_val.shape
test_size = X_test.shape

train_class_count = y_train.nunique()
validation_class_count = y_val.nunique()
test_class_count = y_test.nunique()

In [ ]:
train_size, validation_size, test_size

In [ ]:
train_class_count, validation_class_count, test_class_count

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train.to_frame(), y_train)

In [ ]:
dummy_preds = dummy_model.predict(X_val.to_frame())

In [ ]:
dummy_accuracy = accuracy_score(
    y_val,
    dummy_preds
)

dummy_macro_f1 = f1_score(
    y_val,
    dummy_preds,
    average="macro",
    zero_division=0
)

dummy_weighted_f1 = f1_score(
    y_val,
    dummy_preds,
    average="weighted",
    zero_division=0
)

print(f"Accuracy:    {dummy_accuracy:.4f}")
print(f"Macro-F1:    {dummy_macro_f1:.4f}")
print(f"Weighted-F1: {dummy_weighted_f1:.4f}")

#E1 model

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

In [ ]:
nb_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(
        ngram_range=(1,1),
        min_df = 2,
        lowercase= False
    )),
    ("classifier", MultinomialNB(alpha=1.0))
])

In [ ]:
nb_pipeline.fit(X_train, y_train)
nb_preds = nb_pipeline.predict(X_val)

In [ ]:
nb_accuracy = accuracy_score(
    y_val,
    nb_preds
)

nb_macro_f1 = f1_score(
    y_val,
    nb_preds,
    average="macro",
    zero_division=0
)

nb_weighted_f1 = f1_score(
    y_val,
    nb_preds,
    average="weighted",
    zero_division=0
)

print(f"Accuracy:    {nb_accuracy:.4f}")
print(f"Macro-F1:    {nb_macro_f1:.4f}")
print(f"Weighted-F1: {nb_weighted_f1:.4f}")

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(y_val, nb_preds, digits=3, zero_division=0))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
nb_pipeline.classes_

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_val,
nb_preds,
normalize="true",
xticks_rotation=90,
values_format=".2f")
plt.show()

In [ ]:
X_val.head()

In [ ]:
y_val

In [ ]:
import pandas as pd
error_df = pd.DataFrame({
    "text": X_val.to_numpy(),
    "true_label": y_val.to_numpy(),
    "predicted_label": nb_preds
})

error_df.head()

In [ ]:
request_to_donation = error_df[
    (error_df["true_label"] == "requests_or_urgent_needs")
    & (
        error_df["predicted_label"]
        == "rescue_volunteering_or_donation_effort"
    )
]

print("Error count:", len(request_to_donation))

display(
    request_to_donation.sample(
        n=min(10, len(request_to_donation)),
        random_state=42
    )
)

In [ ]:
missing_to_injured = error_df[
    (error_df["true_label"] == "missing_or_found_people")
    & (
        error_df["predicted_label"]
        == "injured_or_dead_people"
    )
]

print("Error count:", len(missing_to_injured))

display(
    missing_to_injured.sample(
        n=min(10, len(missing_to_injured)),
        random_state=42
    )
)

#E2

In [ ]:
nb_bigram_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(
        ngram_range=(1,2),
        min_df = 2,
        lowercase= False
    )),
    ("classifier", MultinomialNB(alpha=1.0))
])

In [ ]:
nb_bigram_pipeline.fit(X_train, y_train)
nb_bigram_preds = nb_bigram_pipeline.predict(X_val)

In [ ]:
nb_bigram_accuracy = accuracy_score(
    y_val,
    nb_bigram_preds
)

nb_bigram_macro_f1 = f1_score(
    y_val,
    nb_bigram_preds,
    average="macro",
    zero_division=0
)

nb_bigram_weighted_f1 = f1_score(
    y_val,
    nb_bigram_preds,
    average="weighted",
    zero_division=0
)

print(f"Accuracy:    {nb_bigram_accuracy:.4f}")
print(f"Macro-F1:    {nb_bigram_macro_f1:.4f}")
print(f"Weighted-F1: {nb_bigram_weighted_f1:.4f}")

In [ ]:
print(classification_report(y_val, nb_bigram_preds, digits=3, zero_division=0))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_val,
nb_bigram_preds,
normalize="true",
xticks_rotation=90,
values_format=".2f")
plt.show()

#E3

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf_nb_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
        ngram_range=(1,1),
        min_df = 2,
        lowercase= False,
        sublinear_tf=True,
    )),
    ("classifier", MultinomialNB(alpha=1.0))
])

In [ ]:
tfidf_nb_pipeline.fit(X_train, y_train)
tfidf_nb_preds = tfidf_nb_pipeline.predict(X_val)

In [ ]:
idf_accuracy = accuracy_score(
    y_val,
    tfidf_nb_preds
)

idf_macro_f1 = f1_score(
    y_val,
    tfidf_nb_preds,
    average="macro",
    zero_division=0
)
idf_weighted_f1 = f1_score(
    y_val,
    tfidf_nb_preds,
    average="weighted",
    zero_division=0
)

print(f"Accuracy:    {idf_accuracy:.4f}")
print(f"Macro-F1:    {idf_macro_f1:.4f}")
print(f"Weighted-F1: {idf_weighted_f1:.4f}")

In [ ]:
print(classification_report(y_val, tfidf_nb_preds, digits=3, zero_division=0))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_val,
tfidf_nb_preds,
normalize="true",
xticks_rotation=90,
values_format=".2f")
plt.show()

#E4

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
tfidf_lr_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
        ngram_range=(1,1),
        min_df = 2,
        lowercase= False,
        sublinear_tf=True,
    )),
    ("classifier", LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver="liblinear",
    class_weight=None,
    random_state=42
))
])

In [ ]:
tfidf_lr_pipeline.fit(X_train, y_train)
tfidf_lr_preds = tfidf_lr_pipeline.predict(X_val)

In [ ]:
idflr_accuracy = accuracy_score(
    y_val,
    tfidf_lr_preds
)

idflr_macro_f1 = f1_score(
    y_val,
    tfidf_lr_preds,
    average="macro",
    zero_division=0
)
idflr_weighted_f1 = f1_score(
    y_val,
    tfidf_lr_preds,
    average="weighted",
    zero_division=0
)

print(f"Accuracy:    {idflr_accuracy:.4f}")
print(f"Macro-F1:    {idflr_macro_f1:.4f}")
print(f"Weighted-F1: {idflr_weighted_f1:.4f}")

In [ ]:
print(classification_report(y_val, tfidf_lr_preds, digits=3, zero_division=0))

In [ ]:
labels=tfidf_lr_pipeline.classes_

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_val,
tfidf_lr_preds,
normalize="true",
xticks_rotation=90,
labels=tfidf_lr_pipeline.classes_,
values_format=".2f")
plt.show()

#E5

In [ ]:
tfidf_lr_balanced_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
        ngram_range=(1,1),
        min_df = 2,
        lowercase= False,
        sublinear_tf=True,
    )),
    ("classifier", LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver="liblinear",
    class_weight="balanced",
    random_state=42
))
])

In [ ]:
tfidf_lr_balanced_pipeline.fit(X_train, y_train)
tfidf_lr_balanced_preds = tfidf_lr_balanced_pipeline.predict(X_val)

In [ ]:
bidflr_accuracy = accuracy_score(
    y_val,
    tfidf_lr_balanced_preds
)

bidflr_macro_f1 = f1_score(
    y_val,
    tfidf_lr_balanced_preds,
    average="macro",
    zero_division=0
)
bidflr_weighted_f1 = f1_score(
    y_val,
    tfidf_lr_balanced_preds,
    average="weighted",
    zero_division=0
)

print(f"Accuracy:    {bidflr_accuracy:.4f}")
print(f"Macro-F1:    {bidflr_macro_f1:.4f}")
print(f"Weighted-F1: {bidflr_weighted_f1:.4f}")

In [ ]:
print(classification_report(y_val, tfidf_lr_balanced_preds, digits=3, zero_division=0))

#scores

In [ ]:
def evaluate_predictions(
    experiment_name: str,
    y_true,
    y_pred
) -> dict:

    return {
        "experiment": experiment_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )
    }

In [ ]:
e5_result = evaluate_predictions(
    experiment_name="E5_tfidf_lr_balanced",
    y_true=y_val,
    y_pred=tfidf_lr_balanced_preds
)

e5_result

In [ ]:
experiment_results = [
    evaluate_predictions(
        "E0_dummy_most_frequent",
        y_val,
        dummy_preds
    ),
    evaluate_predictions(
        "E1_count_unigram_nb",
        y_val,
        nb_preds
    ),
    evaluate_predictions(
        "E2_count_1_2gram_nb",
        y_val,
        nb_bigram_preds
    ),
    evaluate_predictions(
        "E3_tfidf_unigram_nb",
        y_val,
        tfidf_nb_preds
    ),
    evaluate_predictions(
        "E4_tfidf_unigram_lr",
        y_val,
        tfidf_lr_preds
    ),
    evaluate_predictions(
        "E5_tfidf_unigram_lr_balanced",
        y_val,
        tfidf_lr_balanced_preds
    )
]

In [ ]:
results_df = (
    pd.DataFrame(experiment_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

results_df.round(4)

In [ ]:
RESULTS_PATH = PROJECT_ROOT / "reports" / "validation_results.csv"

results_df.to_csv(
    RESULTS_PATH,
    index=False
)


#E6

In [ ]:
tfidf_bigram_lr_balanced_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
        ngram_range=(1,2),
        min_df = 2,
        lowercase= False,
        sublinear_tf=True,
    )),
    ("classifier", LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver="liblinear",
    class_weight="balanced",
    random_state=42
))
])

In [ ]:
tfidf_bigram_lr_balanced_pipeline.fit(X_train, y_train)
tfidf_bigram_lr_balanced_preds = tfidf_bigram_lr_balanced_pipeline.predict(X_val)

In [ ]:
bbidflr_accuracy = accuracy_score(
    y_val,
    tfidf_bigram_lr_balanced_preds
)

bbidflr_macro_f1 = f1_score(
    y_val,
    tfidf_bigram_lr_balanced_preds,
    average="macro",
    zero_division=0
)
bbidflr_weighted_f1 = f1_score(
    y_val,
    tfidf_bigram_lr_balanced_preds,
    average="weighted",
    zero_division=0
)

print(f"Accuracy:    {bbidflr_accuracy:.4f}")
print(f"Macro-F1:    {bbidflr_macro_f1:.4f}")
print(f"Weighted-F1: {bbidflr_weighted_f1:.4f}")

In [ ]:
print(classification_report(y_val, tfidf_bigram_lr_balanced_preds, digits=3, zero_division=0))

In [ ]:
e6_result = evaluate_predictions(
    "E6_tfidf_1_2gram_lr_balanced",
    y_val,
    tfidf_bigram_lr_balanced_preds
)

experiment_results.append(e6_result)

results_df = (
    pd.DataFrame(experiment_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

results_df.round(4)

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import joblib
import subprocess
import sys
import sklearn


CHECKPOINT_NAME = "after_e6"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / CHECKPOINT_NAME
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
SRC_DIR = PROJECT_ROOT / "src"

for directory in [
    CHECKPOINT_DIR,
    PROCESSED_DATA_DIR,
    MODELS_DIR,
    REPORTS_DIR,
    SRC_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# 1. Preprocessed datasets
train_df.to_parquet(
    PROCESSED_DATA_DIR / "humaid_train_minimal.parquet",
    index=False
)

validation_df.to_parquet(
    PROCESSED_DATA_DIR / "humaid_validation_minimal.parquet",
    index=False
)

test_df.to_parquet(
    PROCESSED_DATA_DIR / "humaid_test_minimal.parquet",
    index=False
)


# 2. Experiment results
results_df.to_csv(
    REPORTS_DIR / "validation_results.csv",
    index=False
)


# 3. Save every fitted model that currently exists
model_names = [
    "dummy_model",
    "nb_pipeline",
    "nb_bigram_pipeline",
    "tfidf_nb_pipeline",
    "tfidf_lr_pipeline",
    "tfidf_lr_balanced_pipeline",
    "tfidf_bigram_lr_balanced_pipeline",
]

saved_models = []

for model_name in model_names:
    if model_name in globals():
        model_path = MODELS_DIR / f"{model_name}.joblib"

        joblib.dump(
            globals()[model_name],
            model_path
        )

        saved_models.append(model_name)
        print(f"Saved model: {model_name}")


# 4. Save checkpoint metadata
best_experiment = (
    results_df
    .sort_values("macro_f1", ascending=False)
    .iloc[0]
    .to_dict()
)

metadata = {
    "checkpoint_name": CHECKPOINT_NAME,
    "created_at": datetime.now().isoformat(),
    "train_rows": len(train_df),
    "validation_rows": len(validation_df),
    "test_rows": len(test_df),
    "text_column": "text_minimal",
    "target_column": "class_label",
    "saved_models": saved_models,
    "best_validation_experiment": best_experiment,
    "python_version": sys.version,
    "scikit_learn_version": sklearn.__version__,
}

with open(
    CHECKPOINT_DIR / "metadata.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        metadata,
        file,
        indent=4,
        ensure_ascii=False
    )


# 5. Save installed package versions
requirements = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True,
    text=True,
    check=True
).stdout

(PROJECT_ROOT / "requirements.txt").write_text(
    requirements,
    encoding="utf-8"
)

print("\nCheckpoint completed successfully.")
print(f"Location: {CHECKPOINT_DIR}")

In [ ]:
for path in sorted(PROJECT_ROOT.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)

        print(
            f"{path.relative_to(PROJECT_ROOT)}"
            f" — {size_mb:.2f} MB"
        )

#E7

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import joblib

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/crisis-text-triage"
)

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

train_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "humaid_train_minimal.parquet"
)

validation_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "humaid_validation_minimal.parquet"
)

test_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "humaid_test_minimal.parquet"
)

results_df = pd.read_csv(
    REPORTS_DIR / "validation_results.csv"
)

tfidf_bigram_lr_balanced_pipeline = joblib.load(
    MODELS_DIR / "tfidf_bigram_lr_balanced_pipeline.joblib"
)

X_train = train_df["text_minimal"]
y_train = train_df["class_label"]

X_val = validation_df["text_minimal"]
y_val = validation_df["class_label"]

X_test = test_df["text_minimal"]
y_test = test_df["class_label"]


In [ ]:
from sklearn.svm import LinearSVC

In [ ]:
tfidf_bigram_svc_balanced_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
        ngram_range=(1,2),
        min_df = 2,
        lowercase= False,
        sublinear_tf=True,
    )),
    ("classifier", LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=5000,
    random_state=42
))
])

In [ ]:
tfidf_bigram_svc_balanced_pipeline.fit(X_train, y_train)
tfidf_bigram_svc_balanced_preds = tfidf_bigram_svc_balanced_pipeline.predict(X_val)

In [ ]:
sbidflr_accuracy = accuracy_score(
    y_val,
    tfidf_bigram_svc_balanced_preds
)

sbidflr_macro_f1 = f1_score(
    y_val,
    tfidf_bigram_svc_balanced_preds,
    average="macro",
    zero_division=0
)
sbidflr_weighted_f1 = f1_score(
    y_val,
    tfidf_bigram_svc_balanced_preds,
    average="weighted",
    zero_division=0
)

print(f"Accuracy:    {sbidflr_accuracy:.4f}")
print(f"Macro-F1:    {sbidflr_macro_f1:.4f}")
print(f"Weighted-F1: {sbidflr_weighted_f1:.4f}")

In [ ]:
print(classification_report(y_val, tfidf_bigram_svc_balanced_preds, digits=3, zero_division=0))

In [ ]:
e7_result = evaluate_predictions(
    "E7_tfidf_1_2gram_linearsvc_balanced",
    y_val,
    tfidf_bigram_svc_balanced_preds
)

experiment_results.append(e7_result)

In [ ]:
experiment_results

In [ ]:
results_df

In [ ]:
results_df = (
    pd.DataFrame(experiment_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

results_df.round(4)

In [ ]:
unique_results = {
    result["experiment"]: result
    for result in experiment_results
}

experiment_results = list(unique_results.values())

results_df = (
    pd.DataFrame(experiment_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

results_df.round(4)

#E8

In [ ]:
char_tfidf_svc_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
      analyzer="char_wb",
      ngram_range=(3, 5),
      min_df=2,
      lowercase=False,
      sublinear_tf=True
    )),
    ("classifier", LinearSVC(
      C=1.0,
      class_weight="balanced",
      max_iter=5000,
      random_state=42
  ))
])

In [ ]:
char_tfidf_svc_pipeline.fit(X_train, y_train)
char_tfidf_svc_preds = char_tfidf_svc_pipeline.predict(X_val)

In [ ]:
wsbidflr_accuracy = accuracy_score(
    y_val,
    char_tfidf_svc_preds
)

wsbidflr_macro_f1 = f1_score(
    y_val,
    char_tfidf_svc_preds,
    average="macro",
    zero_division=0
)
wsbidflr_weighted_f1 = f1_score(
    y_val,
    char_tfidf_svc_preds,
    average="weighted",
    zero_division=0
)

print(f"Accuracy:    {wsbidflr_accuracy:.4f}")
print(f"Macro-F1:    {wsbidflr_macro_f1:.4f}")
print(f"Weighted-F1: {wsbidflr_weighted_f1:.4f}")

In [ ]:
print(classification_report(y_val, char_tfidf_svc_preds, digits=3, zero_division=0))

In [ ]:
e8_result = evaluate_predictions(
    "E8_char_3_5gram_linearsvc_balanced",
    y_val,
    char_tfidf_svc_preds
)

experiment_results.append(e8_result)

In [ ]:
results_df = (
    pd.DataFrame(experiment_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

results_df.round(4)

#E9

In [ ]:
from sklearn.pipeline import FeatureUnion

In [ ]:
e9_pipeline = Pipeline([
    (
        "features",
        FeatureUnion([
            ("word_tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=False)),
            ("char_tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True, lowercase=False)),
        ])
    ),
    ("classifier", LinearSVC(
        C=1.0,
    class_weight="balanced",
    max_iter=5000,
    random_state=42,
    ) )
])

In [ ]:
e9_pipeline.fit(X_train, y_train)
e9_preds = e9_pipeline.predict(X_val)

In [ ]:
wsbidflr9_accuracy = accuracy_score(
    y_val,
    e9_preds
)

wsbidflr9_macro_f1 = f1_score(
    y_val,
    e9_preds,
    average="macro",
    zero_division=0
)
wsbidflr9_weighted_f1 = f1_score(
    y_val,
    e9_preds,
    average="weighted",
    zero_division=0
)

print(f"Accuracy:    {wsbidflr9_accuracy:.4f}")
print(f"Macro-F1:    {wsbidflr9_macro_f1:.4f}")
print(f"Weighted-F1: {wsbidflr9_weighted_f1:.4f}")

In [ ]:
print(classification_report(y_val, e9_preds, digits=3, zero_division=0))

In [ ]:
e9_result = evaluate_predictions(
    "E9_word_char_tfidf_linearsvc_balanced",
    y_val,
    char_tfidf_svc_preds
)

experiment_results.append(e9_result)

In [ ]:
results_df = (
    pd.DataFrame(experiment_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)
results_df.round(4)

#E10 -> E6

In [ ]:
C_VALUES = [0.25, 0.5, 1.0, 2.0, 4.0]

In [ ]:
c_experiment_results = []
c_trained_models = {}

In [ ]:
for c_value in C_VALUES:

    print(f"Training model with C={c_value}...")

    pipeline = Pipeline([
        (
            "vectorizer",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                lowercase=False,
                sublinear_tf=True
            )
        ),
        (
            "classifier",
            LogisticRegression(
                C=c_value,
                max_iter=1000,
                solver="liblinear",
                class_weight="balanced",
                random_state=42
            )
        )
    ])


    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_val)
    report = classification_report(
        y_val,
        predictions,
        output_dict=True,
        zero_division=0
    )

    result = {
        "C": c_value,
        "accuracy": accuracy_score(
            y_val,
            predictions
        ),
        "macro_f1": f1_score(
            y_val,
            predictions,
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            y_val,
            predictions,
            average="weighted",
            zero_division=0
        ),
        "missing_precision": report[
            "missing_or_found_people"
        ]["precision"],
        "missing_recall": report[
            "missing_or_found_people"
        ]["recall"],
        "urgent_precision": report[
            "requests_or_urgent_needs"
        ]["precision"],
        "urgent_recall": report[
            "requests_or_urgent_needs"
        ]["recall"]
    }

    c_experiment_results.append(result)
    c_trained_models[c_value] = pipeline


c_results_df = (
    pd.DataFrame(c_experiment_results)
    .sort_values(
        by="macro_f1",
        ascending=False
    )
    .reset_index(drop=True)
)

display(c_results_df.round(4))

In [ ]:
best_c = c_results_df.loc[0, "C"]

best_c_pipeline = c_trained_models[best_c]

print(f"Best C: {best_c}")
print(
    f"Best Macro-F1: "
    f"{c_results_df.loc[0, 'macro_f1']:.4f}"
)

In [ ]:
best_c_preds = best_c_pipeline.predict(X_val)

print(
    classification_report(
        y_val,
        best_c_preds,
        digits=3,
        zero_division=0
    )
)

In [ ]:
e10_result = evaluate_predictions(
    experiment_name="E10_tfidf_1_2gram_lr_balanced_C2",
    y_true=y_val,
    y_pred=best_c_preds
)

experiment_results = [
    result
    for result in experiment_results
    if result["experiment"] != e10_result["experiment"]
]

experiment_results.append(e10_result)

results_df = (
    pd.DataFrame(experiment_results)
    .drop_duplicates(subset="experiment", keep="last")
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

display(results_df.round(4))

In [ ]:
import joblib

BEST_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "e10_tfidf_bigram_lr_balanced_c2.joblib"
)

C_RESULTS_PATH = (
    PROJECT_ROOT
    / "reports"
    / "c_tuning_results.csv"
)

VALIDATION_RESULTS_PATH = (
    PROJECT_ROOT
    / "reports"
    / "validation_results.csv"
)

joblib.dump(best_c_pipeline, BEST_MODEL_PATH)

c_results_df.to_csv(
    C_RESULTS_PATH,
    index=False
)

results_df.to_csv(
    VALIDATION_RESULTS_PATH,
    index=False
)

print(f"Model saved: {BEST_MODEL_PATH}")
print(f"C results saved: {C_RESULTS_PATH}")
print(f"Experiments saved: {VALIDATION_RESULTS_PATH}")

In [ ]:
best_c_pipeline.named_steps

In [ ]:
vectorizer = best_c_pipeline.named_steps["vectorizer"]

In [ ]:
classifier = best_c_pipeline.named_steps["classifier"]

In [ ]:
feature_names = vectorizer.get_feature_names_out()

In [ ]:
classifier.coef_.shape

In [ ]:
feature_names.shape, classifier.coef_.shape,classifier.classes_.shape

In [ ]:
import numpy as np

In [ ]:
#missing_or_found_people

In [ ]:
index = (np.where(classifier.classes_ == "missing_or_found_people"))[0][0]

In [ ]:
sample_coef = classifier.coef_[index]

In [ ]:
sample_coef.shape

In [ ]:
top_indices = np.argsort(sample_coef)[-15:][::-1]

In [ ]:
top_indices

In [ ]:
(top_indices[0])

In [ ]:
for feature_index in top_indices:
    feature_name = feature_names[feature_index]
    feature_weight = sample_coef[feature_index]

    print(f"{feature_name:<30} {feature_weight:.4f}")

In [ ]:
features_to_inspect = [
    "maryland",
    "california",
    "guardsman",
    "for in"
]

In [ ]:
train_df["text_minimal"].str.contains(features_to_inspect, regex=False, na=False)

In [ ]:
features_to_inspect = [
    "maryland",
    "california",
    "guardsman",
    "for in"
]

for feature in features_to_inspect:
    feature_mask = train_df["text_minimal"].str.contains(
        feature,
        case=False,
        regex=False,
        na=False
    )

    feature_examples = train_df.loc[
        feature_mask,
        ["text_minimal", "class_label"]
    ]

    print("=" * 100)
    print(f"FEATURE: {feature}")
    print(f"TOTAL MESSAGES: {len(feature_examples)}")

    print("\nLABEL COUNTS")
    display(
        feature_examples["class_label"]
        .value_counts()
        .rename("count")
        .to_frame()
    )

    print("\nLABEL PERCENTAGES")
    display(
        feature_examples["class_label"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .rename("percentage")
        .to_frame()
    )

    print("\nRANDOM EXAMPLES")

    if len(feature_examples) > 0:
        display(
            feature_examples.sample(
                n=min(5, len(feature_examples)),
                random_state=42
            )
        )
    else:
        print("No matching messages found.")

    print()

In [ ]:

def get_top_features(
    pipeline,
    class_name: str,
    top_n: int = 15
):

    vectorizer = pipeline.named_steps["vectorizer"]
    classifier = pipeline.named_steps["classifier"]

    feature_names = vectorizer.get_feature_names_out()
    class_names = classifier.classes_

    if class_name not in class_names:
        raise ValueError(
            f"Unknown class: {class_name}\n"
            f"Available classes: {list(class_names)}"
        )

    class_index = np.where(class_names == class_name)[0][0]
    class_weights = classifier.coef_[class_index]

    positive_indices = np.argsort(class_weights)[-top_n:][::-1]
    negative_indices = np.argsort(class_weights)[:top_n]

    positive_features = pd.DataFrame({
        "feature": feature_names[positive_indices],
        "weight": class_weights[positive_indices]
    })

    negative_features = pd.DataFrame({
        "feature": feature_names[negative_indices],
        "weight": class_weights[negative_indices]
    })

    print(f"CLASS: {class_name}")

    print("\nTOP POSITIVE FEATURES")
    display(positive_features.round(4))

    print("\nTOP NEGATIVE FEATURES")
    display(negative_features.round(4))

    return positive_features, negative_features

In [ ]:
missing_positive, missing_negative = get_top_features(
    pipeline=best_c_pipeline,
    class_name="missing_or_found_people",
    top_n=15
)

In [ ]:
urgent_positive, urgent_negative = get_top_features(
    pipeline=best_c_pipeline,
    class_name="requests_or_urgent_needs",
    top_n=15
)

In [ ]:
def extract_all_top_features(pipeline, top_n=15):
    vectorizer = pipeline.named_steps["vectorizer"]
    classifier = pipeline.named_steps["classifier"]

    feature_names = vectorizer.get_feature_names_out()
    records = []

    for class_index, class_name in enumerate(classifier.classes_):
        class_weights = classifier.coef_[class_index]

        positive_indices = np.argsort(class_weights)[-top_n:][::-1]
        negative_indices = np.argsort(class_weights)[:top_n]

        for rank, feature_index in enumerate(positive_indices, start=1):
            records.append({
                "class_name": class_name,
                "direction": "positive",
                "rank": rank,
                "feature": feature_names[feature_index],
                "weight": class_weights[feature_index]
            })

        for rank, feature_index in enumerate(negative_indices, start=1):
            records.append({
                "class_name": class_name,
                "direction": "negative",
                "rank": rank,
                "feature": feature_names[feature_index],
                "weight": class_weights[feature_index]
            })

    return pd.DataFrame(records)

In [ ]:
all_top_features_df = extract_all_top_features(
    pipeline=best_c_pipeline,
    top_n=15
)

In [ ]:
all_top_features_df.shape

In [ ]:
display(all_top_features_df.head(30))

In [ ]:
TOP_FEATURES_PATH = (
    PROJECT_ROOT
    / "reports"
    / "top_features_all_classes.csv"
)

all_top_features_df.to_csv(
    TOP_FEATURES_PATH,
    index=False
)

print(f"Saved: {TOP_FEATURES_PATH}")

In [ ]:
sample_text = "We urgently need food, clean water and medicine."

processed_text = preprocess_text(sample_text)

predicted_class = best_c_pipeline.predict(
    [processed_text]
)[0]

predicted_probabilities = best_c_pipeline.predict_proba(
    [processed_text]
)[0]

predicted_class_index = np.argmax(predicted_probabilities)
predicted_probability = predicted_probabilities[predicted_class_index]

print("Original text:", sample_text)
print("Processed text:", processed_text)
print("Predicted class:", predicted_class)
print("Confidence:", round(predicted_probability, 4))

In [ ]:
processed_text

In [ ]:
sample_vector = vectorizer.transform([processed_text])

class_index = np.where(
    classifier.classes_ == predicted_class
)[0][0]

class_weights = classifier.coef_[class_index]

In [ ]:
active_indices = sample_vector.indices
active_tfidf_values = sample_vector.data

In [ ]:
contributions = (
    active_tfidf_values
    * class_weights[active_indices]
)

In [ ]:
print("Vector shape:", sample_vector.shape)
print("Active feature count:", len(active_indices))
print("Contribution count:", len(contributions))

In [ ]:
local_explanation_df = pd.DataFrame({
    "feature": feature_names[active_indices],
    "tfidf_value": active_tfidf_values,
    "class_weight": class_weights[active_indices],
    "contribution": contributions ,
})

In [ ]:
local_explanation_df = (
    local_explanation_df
    .sort_values("contribution", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
display(local_explanation_df.round(4))

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import joblib


# =========================================================
# CHECKPOINT FOLDERS
# =========================================================

MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# 1. CURRENT BEST MODEL
# =========================================================

best_model_path = MODELS_DIR / "best_tfidf_lr_balanced_c2.joblib"

joblib.dump(
    best_c_pipeline,
    best_model_path
)


# =========================================================
# 2. C=1 MODEL — HIGHER URGENT RECALL ALTERNATIVE
# =========================================================

c1_model_path = None

if "c_trained_models" in globals() and 1.0 in c_trained_models:
    c1_model_path = MODELS_DIR / "tfidf_lr_balanced_c1.joblib"

    joblib.dump(
        c_trained_models[1.0],
        c1_model_path
    )


# =========================================================
# 3. EXPERIMENT RESULTS
# =========================================================

if "results_df" in globals():
    results_df.to_csv(
        REPORTS_DIR / "validation_results.csv",
        index=False
    )

if "c_results_df" in globals():
    c_results_df.to_csv(
        REPORTS_DIR / "c_tuning_results.csv",
        index=False
    )


# =========================================================
# 4. GLOBAL INTERPRETABILITY
# =========================================================

if "all_top_features_df" in globals():
    all_top_features_df.to_csv(
        REPORTS_DIR / "top_features_all_classes.csv",
        index=False
    )


# =========================================================
# 5. LOCAL EXPLANATION SAMPLE
# =========================================================

if "local_explanation_df" in globals():
    local_explanation_df.to_csv(
        REPORTS_DIR / "sample_local_explanation.csv",
        index=False
    )


# =========================================================
# 6. PROJECT STATUS / METADATA
# =========================================================

checkpoint_metadata = {
    "saved_at": datetime.now().isoformat(timespec="seconds"),

    "current_best_experiment": "E10_tfidf_word_1_2_lr_balanced_c2",

    "best_c": float(best_c) if "best_c" in globals() else 2.0,
    "validation_macro_f1": 0.7247,
    "validation_accuracy": 0.746,

    "important_tradeoff": {
        "c_2": {
            "macro_f1": 0.7247,
            "missing_recall": 0.667,
            "urgent_recall": 0.602
        },
        "c_1": {
            "macro_f1": 0.7195,
            "missing_recall": 0.667,
            "urgent_recall": 0.644
        }
    },

    "interpretability_completed": [
        "global positive and negative features",
        "event-specific shortcut inspection",
        "single-message local contribution analysis"
    ],

    "sample_prediction": {
        "text": sample_text if "sample_text" in globals() else None,
        "processed_text": (
            processed_text if "processed_text" in globals() else None
        ),
        "predicted_class": (
            predicted_class if "predicted_class" in globals() else None
        ),
        "confidence": (
            float(predicted_probability)
            if "predicted_probability" in globals()
            else None
        )
    },

    "next_step": (
        "Create a reusable local explanation function, "
        "then continue controlled preprocessing and TF-IDF ablation experiments."
    ),

    "test_set_status": "UNTOUCHED"
}

metadata_path = CHECKPOINTS_DIR / "checkpoint_after_interpretability.json"

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(
        checkpoint_metadata,
        file,
        indent=4,
        ensure_ascii=False
    )


# =========================================================
# 7. QUICK RESTORE FILE
# =========================================================

restore_code = """
import json
import joblib
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/crisis-text-triage")

best_c_pipeline = joblib.load(
    PROJECT_ROOT / "models" / "best_tfidf_lr_balanced_c2.joblib"
)

results_df = pd.read_csv(
    PROJECT_ROOT / "reports" / "validation_results.csv"
)

c_results_df = pd.read_csv(
    PROJECT_ROOT / "reports" / "c_tuning_results.csv"
)

all_top_features_df = pd.read_csv(
    PROJECT_ROOT / "reports" / "top_features_all_classes.csv"
)

local_explanation_df = pd.read_csv(
    PROJECT_ROOT / "reports" / "sample_local_explanation.csv"
)

with open(
    PROJECT_ROOT / "checkpoints" /
    "checkpoint_after_interpretability.json",
    "r",
    encoding="utf-8"
) as file:
    checkpoint_metadata = json.load(file)

vectorizer = best_c_pipeline.named_steps["vectorizer"]
classifier = best_c_pipeline.named_steps["classifier"]
feature_names = vectorizer.get_feature_names_out()

print("Checkpoint restored.")
print("Best model:", checkpoint_metadata["current_best_experiment"])
print("Next step:", checkpoint_metadata["next_step"])
"""

restore_path = CHECKPOINTS_DIR / "restore_after_interpretability.py"

with open(restore_path, "w", encoding="utf-8") as file:
    file.write(restore_code.strip())


# =========================================================
# SUMMARY
# =========================================================

print("CHECKPOINT SAVED SUCCESSFULLY")
print("-" * 60)
print("Best model:", best_model_path)

if c1_model_path is not None:
    print("C=1 alternative:", c1_model_path)

print("Metadata:", metadata_path)
print("Restore script:", restore_path)
print("Test set remains untouched.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%run /content/drive/MyDrive/crisis-text-triage/checkpoints/restore_after_interpretability.py

In [ ]:
import numpy as np
import pandas as pd
def explain_prediction(
    text,
    pipeline,
    preprocess_fn,
    top_n=10
):
    processed_text = preprocess_fn(text)

    vectorizer = pipeline.named_steps["vectorizer"]
    classifier = pipeline.named_steps["classifier"]

    feature_names = vectorizer.get_feature_names_out()
    class_names = classifier.classes_




    probabilities = pipeline.predict_proba(
        [processed_text]
    )[0]

    predicted_class_index = np.argmax(probabilities)

    predicted_class = class_names[predicted_class_index]

    predicted_probability = probabilities[
        predicted_class_index
    ]


    class_probabilities_df = pd.DataFrame({
        "class_name": class_names,
        "probability": probabilities
    })

    class_probabilities_df = (
        class_probabilities_df
        .sort_values(
            "probability",
            ascending=False
        )
        .reset_index(drop=True)
    )

    class_probabilities_df["probability_percent"] = (
        class_probabilities_df["probability"] * 100
    )


    sample_vector = vectorizer.transform(
        [processed_text]
    )

    active_indices = sample_vector.indices
    active_tfidf_values = sample_vector.data

    class_weights = classifier.coef_[
        predicted_class_index
    ]



    contributions = (
        active_tfidf_values
        * class_weights[active_indices]
    )

    local_explanation_df = pd.DataFrame({
        "feature": feature_names[active_indices],
        "tfidf_value": active_tfidf_values,
        "class_weight": class_weights[active_indices],
        "contribution": contributions
    })

    local_explanation_df = (
        local_explanation_df
        .sort_values(
            "contribution",
            ascending=False
        )
        .reset_index(drop=True)
    )


    positive_contributions_df = (
        local_explanation_df[
            local_explanation_df["contribution"] > 0
        ]
        .head(top_n)
        .reset_index(drop=True)
    )


    negative_contributions_df = (
        local_explanation_df[
            local_explanation_df["contribution"] < 0
        ]
        .sort_values(
            "contribution",
            ascending=True
        )
        .head(top_n)
        .reset_index(drop=True)
    )



    print("=" * 80)
    print("PREDICTION EXPLANATION")
    print("=" * 80)

    print("\nOriginal text:")
    print(text)

    print("\nProcessed text:")
    print(processed_text)

    print("\nPredicted class:")
    print(predicted_class)

    print("\nConfidence:")
    print(f"{predicted_probability:.2%}")

    print("\nTOP CLASS PROBABILITIES")
    display(
        class_probabilities_df.head(5).round(4)
    )

    print("\nTOP SUPPORTING FEATURES")

    if len(positive_contributions_df) > 0:
        display(
            positive_contributions_df.round(4)
        )
    else:
        print("No positive contributions found.")

    print("\nTOP OPPOSING FEATURES")

    if len(negative_contributions_df) > 0:
        display(
            negative_contributions_df.round(4)
        )
    else:
        print("No negative contributions found.")



    return {
        "original_text": text,
        "processed_text": processed_text,
        "predicted_class": predicted_class,
        "confidence": float(predicted_probability),
        "class_probabilities": class_probabilities_df,
        "all_contributions": local_explanation_df,
        "positive_contributions": positive_contributions_df,
        "negative_contributions": negative_contributions_df
    }

In [ ]:
import re

In [ ]:
explanation_result = explain_prediction(
    text="Families urgently need clean water, food and medical supplies.",
    pipeline=best_c_pipeline,
    preprocess_fn=preprocess_text,
    top_n=10
)

In [ ]:
explanation_result["predicted_class"]

In [ ]:
explanation_result["confidence"]

In [ ]:
explanation_result["positive_contributions"]

#validation error analysis

In [ ]:
# =========================================================
# CRISISTEXT — SESSION RESTORE
# Training yapılmaz, kayıtlı model ve veriler yüklenir.
# =========================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd


# =========================================================
# 1. PROJECT PATHS
# =========================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/crisis-text-triage"
)

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"


# =========================================================
# 2. HELPER: FIND SAVED PARQUET FILE
# =========================================================

def load_parquet_file(candidate_names):
    for file_name in candidate_names:
        file_path = PROCESSED_DATA_DIR / file_name

        if file_path.exists():
            print(f"Loaded: {file_path}")
            return pd.read_parquet(file_path)

    raise FileNotFoundError(
        "None of these files were found:\n"
        + "\n".join(candidate_names)
    )


# =========================================================
# 3. LOAD PROCESSED DATA
# =========================================================

train_df = load_parquet_file([
    "humaid_train.parquet",
    "humaid_train_minimal.parquet"
])

validation_df = load_parquet_file([
    "humaid_validation.parquet",
    "humaid_validation_minimal.parquet",
    "val.parquet"
])


# =========================================================
# 4. CREATE X / Y VARIABLES
# =========================================================

X_train = train_df["text_minimal"]
y_train = train_df["class_label"]

X_val = validation_df["text_minimal"]
y_val = validation_df["class_label"]


# =========================================================
# 5. LOAD BEST TRAINED MODEL
# =========================================================

BEST_MODEL_PATH = (
    MODELS_DIR
    / "best_tfidf_lr_balanced_c2.joblib"
)

best_c_pipeline = joblib.load(
    BEST_MODEL_PATH
)

vectorizer = best_c_pipeline.named_steps["vectorizer"]
classifier = best_c_pipeline.named_steps["classifier"]
feature_names = vectorizer.get_feature_names_out()


# =========================================================
# 6. LOAD SAVED REPORTS
# =========================================================

validation_results_path = (
    REPORTS_DIR / "validation_results.csv"
)

c_results_path = (
    REPORTS_DIR / "c_tuning_results.csv"
)

top_features_path = (
    REPORTS_DIR / "top_features_all_classes.csv"
)


if validation_results_path.exists():
    results_df = pd.read_csv(validation_results_path)

if c_results_path.exists():
    c_results_df = pd.read_csv(c_results_path)

if top_features_path.exists():
    all_top_features_df = pd.read_csv(top_features_path)


# =========================================================
# 7. LOAD CHECKPOINT METADATA
# =========================================================

metadata_path = (
    CHECKPOINTS_DIR
    / "checkpoint_after_interpretability.json"
)

if metadata_path.exists():
    with open(
        metadata_path,
        "r",
        encoding="utf-8"
    ) as file:
        checkpoint_metadata = json.load(file)
else:
    checkpoint_metadata = {}


# =========================================================
# 8. VERIFY RESTORE
# =========================================================

print("\n" + "=" * 70)
print("SESSION RESTORED — NO TRAINING PERFORMED")
print("=" * 70)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Feature count:", len(feature_names))
print("Classes:", len(classifier.classes_))
print("Model:", BEST_MODEL_PATH.name)

if checkpoint_metadata:
    print(
        "Next step:",
        checkpoint_metadata.get("next_step")
    )

In [ ]:
# =========================================================
# VALIDATION ERROR ANALYSIS
# =========================================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# ---------------------------------------------------------
# 1. VALIDATION PREDICTIONS
# ---------------------------------------------------------

val_predictions = best_c_pipeline.predict(X_val)

val_probabilities = best_c_pipeline.predict_proba(X_val)

class_names = best_c_pipeline.named_steps[
    "classifier"
].classes_


# ---------------------------------------------------------
# 2. CONFIDENCE AND TOP-2 MARGIN
# ---------------------------------------------------------

sorted_probabilities = np.sort(
    val_probabilities,
    axis=1
)

val_confidences = sorted_probabilities[:, -1]
second_best_probabilities = sorted_probabilities[:, -2]

confidence_margins = (
    val_confidences
    - second_best_probabilities
)


# ---------------------------------------------------------
# 3. CREATE ANALYSIS DATAFRAME
# ---------------------------------------------------------

val_analysis_df = pd.DataFrame({
    "text": X_val.to_numpy(),
    "true_label": y_val.to_numpy(),
    "predicted_label": val_predictions,
    "confidence": val_confidences,
    "second_best_probability": second_best_probabilities,
    "confidence_margin": confidence_margins
})

val_analysis_df["is_correct"] = (
    val_analysis_df["true_label"]
    == val_analysis_df["predicted_label"]
)


# ---------------------------------------------------------
# 4. OVERALL VALIDATION METRICS
# ---------------------------------------------------------

validation_accuracy = accuracy_score(
    y_val,
    val_predictions
)

validation_macro_f1 = f1_score(
    y_val,
    val_predictions,
    average="macro",
    zero_division=0
)

validation_weighted_f1 = f1_score(
    y_val,
    val_predictions,
    average="weighted",
    zero_division=0
)

correct_count = int(
    val_analysis_df["is_correct"].sum()
)

wrong_count = int(
    (~val_analysis_df["is_correct"]).sum()
)

print("=" * 70)
print("VALIDATION SUMMARY")
print("=" * 70)

print(f"Total samples:       {len(val_analysis_df)}")
print(f"Correct predictions: {correct_count}")
print(f"Wrong predictions:   {wrong_count}")
print(f"Accuracy:            {validation_accuracy:.4f}")
print(f"Macro-F1:            {validation_macro_f1:.4f}")
print(f"Weighted-F1:         {validation_weighted_f1:.4f}")


# ---------------------------------------------------------
# 5. WRONG PREDICTIONS
# ---------------------------------------------------------

wrong_predictions_df = (
    val_analysis_df[
        ~val_analysis_df["is_correct"]
    ]
    .sort_values(
        ["confidence", "confidence_margin"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nHIGH-CONFIDENCE WRONG PREDICTIONS")

display(
    wrong_predictions_df[
        [
            "text",
            "true_label",
            "predicted_label",
            "confidence",
            "confidence_margin"
        ]
    ]
    .head(20)
    .round(4)
)


# ---------------------------------------------------------
# 6. MOST COMMON CONFUSION PAIRS
# ---------------------------------------------------------

confusion_pairs_df = (
    wrong_predictions_df
    .groupby(
        ["true_label", "predicted_label"]
    )
    .agg(
        error_count=("text", "size"),
        average_confidence=("confidence", "mean"),
        average_margin=("confidence_margin", "mean")
    )
    .reset_index()
    .sort_values(
        "error_count",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nMOST COMMON CONFUSION PAIRS")

display(
    confusion_pairs_df
    .head(20)
    .round(4)
)


# ---------------------------------------------------------
# 7. CLASSIFICATION REPORT
# ---------------------------------------------------------

validation_report_df = pd.DataFrame(
    classification_report(
        y_val,
        val_predictions,
        labels=class_names,
        output_dict=True,
        zero_division=0
    )
).transpose()

print("\nCLASSIFICATION REPORT")

display(
    validation_report_df.round(4)
)


# ---------------------------------------------------------
# 8. SAVE REPORTS
# ---------------------------------------------------------

val_analysis_df.to_csv(
    REPORTS_DIR / "validation_prediction_analysis.csv",
    index=False
)

wrong_predictions_df.to_csv(
    REPORTS_DIR / "validation_wrong_predictions.csv",
    index=False
)

confusion_pairs_df.to_csv(
    REPORTS_DIR / "validation_confusion_pairs.csv",
    index=False
)

validation_report_df.to_csv(
    REPORTS_DIR / "validation_classification_report.csv"
)

print("\nReports saved successfully.")
print("Training was not performed.")
print("Test set remains untouched.")

In [ ]:
val_predictions = best_c_pipeline.predict(X_val)

val_probabilities = best_c_pipeline.predict_proba(X_val)

val_confidences = val_probabilities.max(axis=1)

val_error_df = pd.DataFrame({
    "text": X_val.to_numpy(),
    "true_label": y_val.to_numpy(),
    "predicted_label": val_predictions,
    "confidence": val_confidences
})

val_error_df["is_correct"] = (
    val_error_df["true_label"]
    == val_error_df["predicted_label"]
)

print("Total validation samples:", len(val_error_df))
print("Correct predictions:", val_error_df["is_correct"].sum())
print("Wrong predictions:", (~val_error_df["is_correct"]).sum())

In [ ]:
val_error_df

In [ ]:
wrong_predictions_df = (
    val_error_df[
        val_error_df["is_correct"] == False
    ]
    .sort_values(
        "confidence",
        ascending=False
    )
    .reset_index(drop=True)
)

In [ ]:
wrong_predictions_df

In [ ]:
confusion_pairs_df = (
    wrong_predictions_df
    .groupby([
        "true_label",
        "predicted_label"
    ])
    .size()
    .reset_index(name="error_count")
    .sort_values(
        "error_count",
        ascending=False
    )
    .reset_index(drop=True)
)

In [ ]:
display(confusion_pairs_df.head(20))

In [ ]:
val_error_df.head()

In [ ]:
class_error_summary_df = val_error_df.groupby("true_label").agg(
    total_samples=("is_correct", "size"),
    wrong_predictions=("is_correct", lambda values: (~values).sum())
).reset_index()

In [ ]:
class_error_summary_df["error_rate"] = (
    class_error_summary_df["wrong_predictions"]
    / class_error_summary_df["total_samples"]
)

In [ ]:
class_error_summary_df.sort_values("error_rate", ascending=False)

In [ ]:
other_errors_df = wrong_predictions_df[
    wrong_predictions_df["true_label"] == "other_relevant_information"
]

In [ ]:
other_errors_df

In [ ]:
other_error_distribution_df = (
    other_errors_df
    .groupby("predicted_label")
    .agg(
        error_count=("text", "size"),
        average_confidence=("confidence", "mean")
    )
    .reset_index()
)

In [ ]:
  other_error_distribution_df["error_percentage"] = (
    other_error_distribution_df["error_count"]
    / len(other_errors_df)
    * 100
)

In [ ]:
other_error_distribution_df = (
    other_error_distribution_df
    .sort_values("error_count", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
display(other_error_distribution_df.round(4))

In [ ]:
high_confidence_threshold = 0.70

In [ ]:
other_confidence_summary_df = (
    other_errors_df
    .groupby("predicted_label")
    .agg(
        total_errors=("text", "size"),
        average_confidence=("confidence", "mean"),
        high_confidence_errors=(
            "confidence",
            lambda values: (values >= high_confidence_threshold).sum()
        )
    )
    .reset_index()
)

In [ ]:
other_confidence_summary_df["high_confidence_rate"] = (
    other_confidence_summary_df["high_confidence_errors"]
    / other_confidence_summary_df["total_errors"]
)

In [ ]:
other_confidence_summary_df = (
    other_confidence_summary_df
    .sort_values(
        "high_confidence_rate",
        ascending=False
    )
    .reset_index(drop=True)
)

In [ ]:
display(other_confidence_summary_df.round(4))

In [ ]:
audit_targets = [
    "sympathy_and_support",
    "infrastructure_and_utility_damage",
    "rescue_volunteering_or_donation_effort"
]

audit_samples = []

In [ ]:
for target_label in audit_targets:
    target_examples = (
        other_errors_df[
            other_errors_df["predicted_label"] == target_label
        ]
        .sort_values("confidence", ascending=False)
        .head(5)
        .copy()
    )

    audit_samples.append(target_examples)

In [ ]:
manual_audit_df = pd.concat(
    audit_samples,
    ignore_index=True
)

In [ ]:
manual_audit_df["audit_decision"] = ""
manual_audit_df["audit_note"] = ""

In [ ]:
display(
    manual_audit_df[
        [
            "text",
            "true_label",
            "predicted_label",
            "confidence",
            "audit_decision",
            "audit_note"
        ]
    ]
)

In [ ]:
audit_results = {
    0: {
        "audit_decision": "likely_mislabeled",
        "audit_note": "Explicit sympathy language: 'our thoughts are with all affected'."
    },
    1: {
        "audit_decision": "likely_mislabeled",
        "audit_note": "Explicit support language: 'my thoughts are with my family and friends'."
    },
    2: {
        "audit_decision": "ambiguous_label",
        "audit_note": "Contains both sympathy ('our thoughts go out') and caution ('be safe')."
    },
    3: {
        "audit_decision": "likely_mislabeled",
        "audit_note": "Directly expresses sympathy toward Florida and hurricane victims."
    },
    4: {
        "audit_decision": "ambiguous_label",
        "audit_note": "Mixed humorous/personal content with sympathy and safety language."
    },
    5: {
        "audit_decision": "likely_mislabeled",
        "audit_note": "Direct report about damage left behind by Hurricane Maria."
    },
    6: {
        "audit_decision": "likely_mislabeled",
        "audit_note": "Explicit flood damage update."
    },
    7: {
        "audit_decision": "likely_mislabeled",
        "audit_note": "Describes and shows hurricane damage."
    },
    8: {
        "audit_decision": "likely_mislabeled",
        "audit_note": "Reports cleanup following hurricane damage."
    },
    9: {
        "audit_decision": "ambiguous_label",
        "audit_note": "Reports major crop destruction; clearly disaster damage, but infrastructure label is not perfectly exact."
    },
    10: {
        "audit_decision": "likely_mislabeled",
        "audit_note": "Organization is preparing to assist in hurricane relief efforts."
    },
    11: {
        "audit_decision": "likely_mislabeled",
        "audit_note": "Explicit collection of items for Kerala flood relief."
    },
    12: {
        "audit_decision": "likely_mislabeled",
        "audit_note": "Announces a disaster relief fund for affected people."
    },
    13: {
        "audit_decision": "likely_mislabeled",
        "audit_note": "Provides bank and payment details for cyclone relief donations."
    },
    14: {
        "audit_decision": "likely_mislabeled",
        "audit_note": "Explicit fundraising and donation request."
    }
}

In [ ]:
for row_index, result in audit_results.items():
    manual_audit_df.loc[
        row_index,
        "audit_decision"
    ] = result["audit_decision"]

    manual_audit_df.loc[
        row_index,
        "audit_note"
    ] = result["audit_note"]

In [ ]:
display(
    manual_audit_df[
        [
            "text",
            "true_label",
            "predicted_label",
            "confidence",
            "audit_decision",
            "audit_note"
        ]
    ]
)

In [ ]:
audit_summary_df = (
    manual_audit_df["audit_decision"]
    .value_counts()
    .rename_axis("audit_decision")
    .reset_index(name="sample_count")
)

audit_summary_df["percentage"] = (
    audit_summary_df["sample_count"]
    / len(manual_audit_df)
    * 100
)

display(audit_summary_df.round(2))

In [ ]:
manual_audit_df.to_csv(
    REPORTS_DIR / "manual_error_audit.csv",
    index=False
)

audit_summary_df.to_csv(
    REPORTS_DIR / "manual_error_audit_summary.csv",
    index=False
)

print("Manual audit reports saved.")

#preprocessing ablation

##E11

In [ ]:
X_train_raw = train_df["tweet_text"]
X_val_raw = validation_df["tweet_text"]

In [ ]:
print(X_train_raw.shape)
print(X_val_raw.shape)

print("\nRAW:")
print(X_train_raw.iloc[0])

print("\nMINIMAL:")
print(X_train.iloc[0])

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [ ]:
e11_raw_pipeline = Pipeline([
    (
        "vectorizer",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            lowercase=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=2.0,
            class_weight="balanced",
            solver="liblinear",
            max_iter=1000,
            random_state=42
        )
    )
])

In [ ]:
e11_raw_pipeline.fit(X_train_raw, y_train)

In [ ]:
e11_raw_predictions = e11_raw_pipeline.predict(
    X_val_raw
)

In [ ]:
e11_result = evaluate_predictions(
    experiment_name="E11_raw_text_tfidf_1_2_lr_balanced_C2",
    y_true=y_val,
    y_pred=e11_raw_predictions
)

In [ ]:
e11_result

In [ ]:
e10_predictions = best_c_pipeline.predict(X_val)

e10_result = evaluate_predictions(
    experiment_name="E10_minimal_text_tfidf_1_2_lr_balanced_C2",
    y_true=y_val,
    y_pred=e10_predictions
)

In [ ]:
ablation_comparison_df = pd.DataFrame([
    e10_result,
    e11_result
])

display(
    ablation_comparison_df.round(4).sort_values(
        "accuracy",
        ascending=False
    )
)

In [ ]:
from sklearn.metrics import classification_report

e10_report = classification_report(
    y_val,
    e10_predictions,
    output_dict=True,
    zero_division=0
)

e11_report = classification_report(
    y_val,
    e11_raw_predictions,
    output_dict=True,
    zero_division=0
)

critical_recall_comparison_df = pd.DataFrame([
    {
        "experiment": "E10_minimal",
        "missing_recall": e10_report["missing_or_found_people"]["recall"],
        "urgent_recall": e10_report["requests_or_urgent_needs"]["recall"]
    },
    {
        "experiment": "E11_raw",
        "missing_recall": e11_report["missing_or_found_people"]["recall"],
        "urgent_recall": e11_report["requests_or_urgent_needs"]["recall"]
    }
])

display(critical_recall_comparison_df.round(4))

In [ ]:
VALIDATION_RESULTS_PATH = (
    REPORTS_DIR
    / "validation_results.csv"
)

results_df = pd.read_csv(
    VALIDATION_RESULTS_PATH
)

experiment_results = results_df.to_dict(
    orient="records"
)

print("Loaded experiments:", len(experiment_results))
display(results_df.round(4))

In [ ]:
experiment_results.append(e11_result)

results_df = (
    pd.DataFrame(experiment_results)
    .drop_duplicates(subset="experiment", keep="last")
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

display(results_df.round(4))

In [ ]:
results_df.to_csv(
    VALIDATION_RESULTS_PATH,
    index=False
)

print("Validation results updated:")
print(VALIDATION_RESULTS_PATH)

#E12

In [ ]:
e12_stopwords_pipeline = Pipeline([
    (
        "vectorizer",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            lowercase=True,
            stop_words="english"
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=2.0,
            class_weight="balanced",
            solver="liblinear",
            max_iter=1000,
            random_state=42
        )
    )
])

In [ ]:
e12_stopwords_pipeline.fit(X_train_raw, y_train)
e12_predictions = e12_stopwords_pipeline.predict(X_val_raw)

In [ ]:
e12_result = evaluate_predictions(
    experiment_name="E12_raw_stopwords_tfidf_1_2_lr_balanced_C2",
    y_true=y_val,
    y_pred=e12_predictions
)

In [ ]:
e12_result

In [ ]:
e12_report = classification_report(
    y_val,
    e12_predictions,
    output_dict=True,
    zero_division=0
)

print(
    "Missing recall:",
    round(e12_report["missing_or_found_people"]["recall"], 4)
)

print(
    "Urgent recall:",
    round(e12_report["requests_or_urgent_needs"]["recall"], 4)
)

In [ ]:
experiment_results = [
    result
    for result in experiment_results
    if result["experiment"] != e12_result["experiment"]
]

experiment_results.append(e12_result)

results_df = (
    pd.DataFrame(experiment_results)
    .drop_duplicates(subset="experiment", keep="last")
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

results_df.to_csv(
    VALIDATION_RESULTS_PATH,
    index=False
)

display(results_df.round(4))

#E13

In [ ]:
e13_min_df5_pipeline = Pipeline([
    (
        "vectorizer",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=5,
            sublinear_tf=True,
            lowercase=True,
            stop_words=None
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=2.0,
            class_weight="balanced",
            solver="liblinear",
            max_iter=1000,
            random_state=42
        )
    )
])

In [ ]:
e13_min_df5_pipeline.fit(X_train_raw, y_train)
e13_predictions = e13_min_df5_pipeline.predict(X_val_raw)

In [ ]:
e13_result = evaluate_predictions(
    experiment_name="E13_raw_min_df5_tfidf_1_2_lr_balanced_C2",
    y_true=y_val,
    y_pred=e13_predictions
)

In [ ]:
e12_result

In [ ]:
e13_report = classification_report(
    y_val,
    e13_predictions,
    output_dict=True,
    zero_division=0
)

print(
    "Missing recall:",
    round(e13_report["missing_or_found_people"]["recall"], 4)
)

print(
    "Urgent recall:",
    round(e13_report["requests_or_urgent_needs"]["recall"], 4)
)

In [ ]:
experiment_results = [
    result
    for result in experiment_results
    if result["experiment"] != e13_result["experiment"]
]

experiment_results.append(e13_result)

results_df = (
    pd.DataFrame(experiment_results)
    .drop_duplicates(subset="experiment", keep="last")
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

results_df.to_csv(
    VALIDATION_RESULTS_PATH,
    index=False
)

display(results_df.round(4))

#E14

In [ ]:
e14_min_df1_pipeline = Pipeline([
    (
        "vectorizer",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=1,
            sublinear_tf=True,
            lowercase=True,
            stop_words=None
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=2.0,
            class_weight="balanced",
            solver="liblinear",
            max_iter=1000,
            random_state=42
        )
    )
])

In [ ]:
e14_min_df1_pipeline.fit(X_train_raw, y_train)
e14_predictions = e14_min_df1_pipeline.predict(X_val_raw)

In [ ]:
e14_result = evaluate_predictions(
    experiment_name="E14_raw_min_df1_tfidf_1_2_lr_balanced_C2",
    y_true=y_val,
    y_pred=e14_predictions
)

In [ ]:
e14_result

In [ ]:
e14_report = classification_report(
    y_val,
    e14_predictions,
    output_dict=True,
    zero_division=0
)

print(
    "Missing recall:",
    round(e14_report["missing_or_found_people"]["recall"], 4)
)

print(
    "Urgent recall:",
    round(e14_report["requests_or_urgent_needs"]["recall"], 4)
)

In [ ]:
experiment_results = [
    result
    for result in experiment_results
    if result["experiment"] != e14_result["experiment"]
]

experiment_results.append(e14_result)

results_df = (
    pd.DataFrame(experiment_results)
    .drop_duplicates(subset="experiment", keep="last")
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

results_df.to_csv(
    VALIDATION_RESULTS_PATH,
    index=False
)

display(results_df.round(4))

#E15

In [ ]:
e15_no_sublinear_pipeline = Pipeline([
    (
        "vectorizer",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=False,
            lowercase=True,
            stop_words=None
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=2.0,
            class_weight="balanced",
            solver="liblinear",
            max_iter=1000,
            random_state=42
        )
    )
])


e15_no_sublinear_pipeline.fit(
    X_train_raw,
    y_train
)


e15_predictions = e15_no_sublinear_pipeline.predict(
    X_val_raw
)


e15_result = evaluate_predictions(
    experiment_name="E15_raw_no_sublinear_tfidf_1_2_lr_balanced_C2",
    y_true=y_val,
    y_pred=e15_predictions
)


e15_report = classification_report(
    y_val,
    e15_predictions,
    output_dict=True,
    zero_division=0
)


experiment_results = [
    result
    for result in experiment_results
    if result["experiment"] != e15_result["experiment"]
]

experiment_results.append(e15_result)


results_df = (
    pd.DataFrame(experiment_results)
    .drop_duplicates(
        subset="experiment",
        keep="last"
    )
    .sort_values(
        "macro_f1",
        ascending=False
    )
    .reset_index(drop=True)
)

results_df.to_csv(
    VALIDATION_RESULTS_PATH,
    index=False
)


print("E15 RESULT")
print(e15_result)

print(
    "\nMissing recall:",
    round(
        e15_report["missing_or_found_people"]["recall"],
        4
    )
)

print(
    "Urgent recall:",
    round(
        e15_report["requests_or_urgent_needs"]["recall"],
        4
    )
)

display(results_df.round(4))

#E16

In [ ]:

e16_max_df_pipeline = Pipeline([
    (
        "vectorizer",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            lowercase=True,
            stop_words=None
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=2.0,
            class_weight="balanced",
            solver="liblinear",
            max_iter=1000,
            random_state=42
        )
    )
])


e16_max_df_pipeline.fit(
    X_train_raw,
    y_train
)


e16_predictions = e16_max_df_pipeline.predict(
    X_val_raw
)


e16_result = evaluate_predictions(
    experiment_name="E16_raw_max_df095_tfidf_1_2_lr_balanced_C2",
    y_true=y_val,
    y_pred=e16_predictions
)


e16_report = classification_report(
    y_val,
    e16_predictions,
    output_dict=True,
    zero_division=0
)


experiment_results = [
    result
    for result in experiment_results
    if result["experiment"] != e16_result["experiment"]
]

experiment_results.append(e16_result)


results_df = (
    pd.DataFrame(experiment_results)
    .drop_duplicates(
        subset="experiment",
        keep="last"
    )
    .sort_values(
        "macro_f1",
        ascending=False
    )
    .reset_index(drop=True)
)

results_df.to_csv(
    VALIDATION_RESULTS_PATH,
    index=False
)


print("E16 RESULT")
print(e16_result)

print(
    "\nMissing recall:",
    round(
        e16_report["missing_or_found_people"]["recall"],
        4
    )
)

print(
    "Urgent recall:",
    round(
        e16_report["requests_or_urgent_needs"]["recall"],
        4
    )
)

display(results_df.round(4))

In [ ]:
e11_feature_count = len(
    e11_raw_pipeline.named_steps["vectorizer"].get_feature_names_out()
)

e16_feature_count = len(
    e16_max_df_pipeline.named_steps["vectorizer"].get_feature_names_out()
)

different_predictions = (
    e11_raw_predictions != e16_predictions
).sum()

print("E11 feature count:", e11_feature_count)
print("E16 feature count:", e16_feature_count)
print("Different validation predictions:", different_predictions)

In [ ]:
from sklearn.base import clone
from pathlib import Path
from datetime import datetime
import pandas as pd
import joblib
import json

In [ ]:
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
selected_experiment = (
    "E11_raw_text_tfidf_1_2_lr_balanced_C2"
)

selected_validation_pipeline = e11_raw_pipeline
selected_validation_result = e11_result
selected_validation_report = classification_report(
    y_val,
    e11_raw_predictions,
    output_dict=True,
    zero_division=0
)

In [ ]:
validation_model_path = (
    MODELS_DIR
    / "selected_validation_model_e11.joblib"
)

joblib.dump(
    selected_validation_pipeline,
    validation_model_path
)

In [ ]:
X_train_val_raw = pd.concat(
    [
        X_train_raw,
        X_val_raw
    ],
    ignore_index=True
)

y_train_val = pd.concat(
    [
        y_train,
        y_val
    ],
    ignore_index=True
)

print("Train samples:", len(X_train_raw))
print("Validation samples:", len(X_val_raw))
print("Combined samples:", len(X_train_val_raw))

In [ ]:
final_pipeline = clone(
    selected_validation_pipeline
)

In [ ]:
final_pipeline.fit(
    X_train_val_raw,
    y_train_val
)

In [ ]:
final_model_path = (
    MODELS_DIR
    / "final_e11_train_plus_validation.joblib"
)

joblib.dump(
    final_pipeline,
    final_model_path
)

In [ ]:
final_model_metadata = {
    "saved_at": datetime.now().isoformat(
        timespec="seconds"
    ),

    "selected_experiment": selected_experiment,

    "selection_metric": "validation_macro_f1",

    "validation_metrics": {
        "accuracy": float(
            selected_validation_result["accuracy"]
        ),
        "macro_f1": float(
            selected_validation_result["macro_f1"]
        ),
        "weighted_f1": float(
            selected_validation_result["weighted_f1"]
        ),
        "missing_recall": float(
            selected_validation_report[
                "missing_or_found_people"
            ]["recall"]
        ),
        "urgent_recall": float(
            selected_validation_report[
                "requests_or_urgent_needs"
            ]["recall"]
        )
    },

    "training_data": {
        "train_samples": int(len(X_train_raw)),
        "validation_samples": int(len(X_val_raw)),
        "combined_samples": int(len(X_train_val_raw))
    },

    "vectorizer_configuration": {
        "type": "TfidfVectorizer",
        "ngram_range": [1, 2],
        "min_df": 2,
        "sublinear_tf": True,
        "lowercase": True,
        "stop_words": None
    },

    "classifier_configuration": {
        "type": "LogisticRegression",
        "C": 2.0,
        "class_weight": "balanced",
        "solver": "liblinear",
        "max_iter": 1000,
        "random_state": 42
    },

    "ablation_conclusion": {
        "raw_vs_minimal": (
            "Raw text produced a marginally higher "
            "validation Macro-F1 with identical critical recalls."
        ),
        "stopword_removal": (
            "Reduced overall Macro-F1 and urgent recall."
        ),
        "min_df_1": (
            "Improved urgent recall slightly but reduced "
            "overall Macro-F1."
        ),
        "min_df_5": (
            "Reduced Macro-F1 without improving critical recalls."
        ),
        "sublinear_tf": (
            "Provided a small performance improvement."
        ),
        "max_df_095": (
            "Removed no features and changed no predictions."
        )
    },

    "test_set_status": "UNTOUCHED"
}

metadata_path = (
    CHECKPOINTS_DIR
    / "final_model_selection.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_model_metadata,
        file,
        indent=4,
        ensure_ascii=False
    )

In [ ]:
results_df.to_csv(
    REPORTS_DIR / "validation_results.csv",
    index=False
)

In [ ]:
final_vectorizer = final_pipeline.named_steps[
    "vectorizer"
]

final_classifier = final_pipeline.named_steps[
    "classifier"
]

print("\n" + "=" * 72)
print("FINAL MODEL TRAINING COMPLETED")
print("=" * 72)

print("Selected experiment:", selected_experiment)
print("Validation Macro-F1:", round(
    selected_validation_result["macro_f1"],
    4
))
print("Final training samples:", len(X_train_val_raw))
print(
    "Final feature count:",
    len(final_vectorizer.get_feature_names_out())
)
print("Classes:", len(final_classifier.classes_))

print("\nValidation model saved:")
print(validation_model_path)

print("\nFinal train+validation model saved:")
print(final_model_path)

print("\nMetadata saved:")
print(metadata_path)

print("\nTEST SET STATUS: UNTOUCHED")

In [ ]:
test_df = load_parquet_file([
    "humaid_test.parquet",
    "humaid_test_minimal.parquet"
])

In [ ]:
X_test_raw = test_df["tweet_text"]
y_test = test_df["class_label"]

In [ ]:
test_predictions = final_pipeline.predict(X_test_raw)
test_probabilities = final_pipeline.predict_proba(X_test_raw)
test_confidences = test_probabilities.max(axis=1)

In [ ]:
print("Test samples:", len(X_test_raw))
print("Prediction shape:", test_predictions.shape)
print("Probability shape:", test_probabilities.shape)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

test_macro_f1 = f1_score(
    y_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_weighted_f1 = f1_score(
    y_test,
    test_predictions,
    average="weighted",
    zero_division=0
)

test_report = classification_report(
    y_test,
    test_predictions,
    labels=final_classifier.classes_,
    output_dict=True,
    zero_division=0
)

print("=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print("Accuracy:", round(test_accuracy, 4))
print("Macro-F1:", round(test_macro_f1, 4))
print("Weighted-F1:", round(test_weighted_f1, 4))

print(
    "Missing recall:",
    round(
        test_report["missing_or_found_people"]["recall"],
        4
    )
)

print(
    "Urgent recall:",
    round(
        test_report["requests_or_urgent_needs"]["recall"],
        4
    )
)

In [ ]:
# =========================================================
# SAVE FINAL TEST RESULTS
# =========================================================

from sklearn.metrics import confusion_matrix
from datetime import datetime
import pandas as pd
import numpy as np
import json


# 1. Final metric summary
final_test_metrics = {
    "evaluated_at": datetime.now().isoformat(timespec="seconds"),
    "model": "final_e11_train_plus_validation.joblib",
    "test_samples": int(len(y_test)),
    "accuracy": float(test_accuracy),
    "macro_f1": float(test_macro_f1),
    "weighted_f1": float(test_weighted_f1),
    "missing_recall": float(
        test_report["missing_or_found_people"]["recall"]
    ),
    "urgent_recall": float(
        test_report["requests_or_urgent_needs"]["recall"]
    ),
    "evaluation_policy": (
        "Test set was evaluated once after final model selection."
    )
}


# 2. Classification report
test_report_df = (
    pd.DataFrame(test_report)
    .transpose()
)


# 3. Per-message predictions
sorted_test_probabilities = np.sort(
    test_probabilities,
    axis=1
)

second_best_test_probabilities = (
    sorted_test_probabilities[:, -2]
)

test_prediction_df = pd.DataFrame({
    "text": X_test_raw.to_numpy(),
    "true_label": y_test.to_numpy(),
    "predicted_label": test_predictions,
    "confidence": test_confidences,
    "second_best_probability": second_best_test_probabilities
})

test_prediction_df["confidence_margin"] = (
    test_prediction_df["confidence"]
    - test_prediction_df["second_best_probability"]
)

test_prediction_df["is_correct"] = (
    test_prediction_df["true_label"]
    == test_prediction_df["predicted_label"]
)


# 4. Confusion matrices
class_names = final_classifier.classes_

test_confusion_matrix = confusion_matrix(
    y_test,
    test_predictions,
    labels=class_names
)

test_confusion_matrix_normalized = confusion_matrix(
    y_test,
    test_predictions,
    labels=class_names,
    normalize="true"
)

test_confusion_df = pd.DataFrame(
    test_confusion_matrix,
    index=class_names,
    columns=class_names
)

test_confusion_normalized_df = pd.DataFrame(
    test_confusion_matrix_normalized,
    index=class_names,
    columns=class_names
)


# 5. Save everything
with open(
    REPORTS_DIR / "final_test_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_test_metrics,
        file,
        indent=4,
        ensure_ascii=False
    )

test_report_df.to_csv(
    REPORTS_DIR / "final_test_classification_report.csv"
)

test_prediction_df.to_csv(
    REPORTS_DIR / "final_test_predictions.csv",
    index=False
)

test_confusion_df.to_csv(
    REPORTS_DIR / "final_test_confusion_matrix.csv"
)

test_confusion_normalized_df.to_csv(
    REPORTS_DIR / "final_test_confusion_matrix_normalized.csv"
)


# 6. Update final model metadata
metadata_path = (
    CHECKPOINTS_DIR
    / "final_model_selection.json"
)

with open(
    metadata_path,
    "r",
    encoding="utf-8"
) as file:
    final_metadata = json.load(file)

final_metadata["test_set_status"] = "EVALUATED_ONCE"
final_metadata["final_test_metrics"] = final_test_metrics

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_metadata,
        file,
        indent=4,
        ensure_ascii=False
    )


print("=" * 70)
print("FINAL TEST REPORTS SAVED")
print("=" * 70)

print("Accuracy:", round(test_accuracy, 4))
print("Macro-F1:", round(test_macro_f1, 4))
print("Missing recall:", round(
    test_report["missing_or_found_people"]["recall"],
    4
))
print("Urgent recall:", round(
    test_report["requests_or_urgent_needs"]["recall"],
    4
))

print("\nSaved to:", REPORTS_DIR)
print("Test set status: EVALUATED ONCE — NO FURTHER TUNING")

#CONF MATRIX

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
short_class_names = [
    "caution",
    "displaced",
    "damage",
    "injured/dead",
    "missing",
    "not humanitarian",
    "other info",
    "urgent needs",
    "rescue/donation",
    "sympathy"
]


fig, ax = plt.subplots(
    figsize=(14, 11)
)

ConfusionMatrixDisplay.from_predictions(
    y_true=y_test,
    y_pred=test_predictions,
    labels=class_names,
    display_labels=short_class_names,
    normalize="true",
    values_format=".2f",
    xticks_rotation=45,
    colorbar=True,
    ax=ax
)

ax.set_title(
    "CrisisText — Normalized Test Confusion Matrix",
    fontsize=16,
    pad=20
)

ax.set_xlabel(
    "Predicted label",
    fontsize=12
)

ax.set_ylabel(
    "True label",
    fontsize=12
)

plt.tight_layout()


confusion_plot_path = (
    REPORTS_DIR
    / "final_test_confusion_matrix_normalized.png"
)

plt.savefig(
    confusion_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", confusion_plot_path)

In [ ]:
test_errors_df = test_prediction_df[
    test_prediction_df["is_correct"] == False
].copy()

test_confusion_pairs_df = (
    test_errors_df
    .groupby([
        "true_label",
        "predicted_label"
    ])
    .agg(
        error_count=("text", "size"),
        average_confidence=("confidence", "mean")
    )
    .reset_index()
    .sort_values(
        "error_count",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    test_confusion_pairs_df
    .head(20)
    .round(4)
)

test_confusion_pairs_df.to_csv(
    REPORTS_DIR / "final_test_confusion_pairs.csv",
    index=False
)

print("Saved:", REPORTS_DIR / "final_test_confusion_pairs.csv")

#INFERENCE

In [ ]:
%%writefile /content/drive/MyDrive/crisis-text-triage/src/inference.py

from pathlib import Path
from typing import Any

import joblib
import numpy as np


PROJECT_ROOT = Path(__file__).resolve().parents[1]

DEFAULT_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "final_e11_train_plus_validation.joblib"
)


CLASS_DISPLAY_NAMES = {
    "caution_and_advice": "Caution and Advice",
    "displaced_people_and_evacuations": "Displaced People and Evacuations",
    "infrastructure_and_utility_damage": "Infrastructure and Utility Damage",
    "injured_or_dead_people": "Injured or Dead People",
    "missing_or_found_people": "Missing or Found People",
    "not_humanitarian": "Not Humanitarian",
    "other_relevant_information": "Other Relevant Information",
    "requests_or_urgent_needs": "Requests or Urgent Needs",
    "rescue_volunteering_or_donation_effort": "Rescue, Volunteering or Donation Effort",
    "sympathy_and_support": "Sympathy and Support"
}


def load_model(model_path: str | Path | None = None):
    path = Path(model_path) if model_path else DEFAULT_MODEL_PATH

    if not path.exists():
        raise FileNotFoundError(f"Model file not found: {path}")

    return joblib.load(path)


def predict_message(
    text: str,
    model,
    top_n: int = 8
) -> dict[str, Any]:
    if not isinstance(text, str):
        raise TypeError("text must be a string")

    text = text.strip()

    if not text:
        raise ValueError("text cannot be empty")

    if top_n < 1:
        raise ValueError("top_n must be at least 1")

    vectorizer = model.named_steps["vectorizer"]
    classifier = model.named_steps["classifier"]

    class_names = classifier.classes_
    feature_names = vectorizer.get_feature_names_out()

    probabilities = model.predict_proba([text])[0]
    predicted_class_index = int(np.argmax(probabilities))
    predicted_class = str(class_names[predicted_class_index])
    confidence = float(probabilities[predicted_class_index])

    sorted_class_indices = np.argsort(probabilities)[::-1]

    class_probabilities = [
        {
            "class_name": str(class_names[index]),
            "display_name": CLASS_DISPLAY_NAMES.get(
                str(class_names[index]),
                str(class_names[index])
            ),
            "probability": float(probabilities[index])
        }
        for index in sorted_class_indices
    ]

    text_vector = vectorizer.transform([text])
    active_indices = text_vector.indices
    active_values = text_vector.data

    class_weights = classifier.coef_[predicted_class_index]

    contributions = (
        active_values
        * class_weights[active_indices]
    )

    contribution_records = [
        {
            "feature": str(feature_names[feature_index]),
            "tfidf_value": float(tfidf_value),
            "class_weight": float(class_weights[feature_index]),
            "contribution": float(contribution)
        }
        for feature_index, tfidf_value, contribution in zip(
            active_indices,
            active_values,
            contributions
        )
    ]

    supporting_features = sorted(
        [
            record
            for record in contribution_records
            if record["contribution"] > 0
        ],
        key=lambda record: record["contribution"],
        reverse=True
    )[:top_n]

    opposing_features = sorted(
        [
            record
            for record in contribution_records
            if record["contribution"] < 0
        ],
        key=lambda record: record["contribution"]
    )[:top_n]

    return {
        "text": text,
        "predicted_class": predicted_class,
        "display_name": CLASS_DISPLAY_NAMES.get(
            predicted_class,
            predicted_class
        ),
        "confidence": confidence,
        "class_probabilities": class_probabilities,
        "supporting_features": supporting_features,
        "opposing_features": opposing_features
    }


if __name__ == "__main__":
    model = load_model()

    sample_text = (
        "Families urgently need clean water, food and medical supplies."
    )

    result = predict_message(
        text=sample_text,
        model=model,
        top_n=5
    )

    print("Prediction:", result["display_name"])
    print("Confidence:", f'{result["confidence"]:.2%}')

    print("\nSupporting features:")
    for feature in result["supporting_features"]:
        print(
            feature["feature"],
            round(feature["contribution"], 4)
        )

In [ ]:
import sys

SRC_DIR = "/content/drive/MyDrive/crisis-text-triage/src"

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from inference import load_model, predict_message

model = load_model()

result = predict_message(
    text="Families urgently need food, clean water and medicine.",
    model=model,
    top_n=5
)

print("Prediction:", result["display_name"])
print("Confidence:", f'{result["confidence"]:.2%}')

print("\nTop classes:")

for item in result["class_probabilities"][:3]:
    print(
        item["display_name"],
        f'{item["probability"]:.2%}'
    )

print("\nSupporting features:")

for item in result["supporting_features"]:
    print(
        item["feature"],
        round(item["contribution"], 4)
    )

#UI

In [ ]:
!pip install -q streamlit

In [ ]:
%%writefile /content/drive/MyDrive/crisis-text-triage/app.py

from pathlib import Path
import sys

import pandas as pd
import streamlit as st


PROJECT_ROOT = Path(__file__).resolve().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from inference import load_model, predict_message


st.set_page_config(
    page_title="CrisisText",
    page_icon="🚨",
    layout="wide",
    initial_sidebar_state="expanded"
)


st.markdown(
    """
    <style>
        .block-container {
            max-width: 1200px;
            padding-top: 2rem;
            padding-bottom: 3rem;
        }

        .main-title {
            font-size: 2.8rem;
            font-weight: 800;
            margin-bottom: 0;
        }

        .subtitle {
            font-size: 1.1rem;
            color: #6b7280;
            margin-top: 0.2rem;
            margin-bottom: 2rem;
        }

        .prediction-card {
            border: 1px solid rgba(128, 128, 128, 0.25);
            border-radius: 14px;
            padding: 1.3rem;
            margin-top: 0.5rem;
            margin-bottom: 1rem;
        }

        .prediction-label {
            font-size: 1.6rem;
            font-weight: 750;
            margin-bottom: 0.2rem;
        }

        .small-note {
            color: #6b7280;
            font-size: 0.9rem;
        }
    </style>
    """,
    unsafe_allow_html=True
)


@st.cache_resource
def get_model():
    return load_model()


def build_probability_dataframe(result):
    dataframe = pd.DataFrame(
        result["class_probabilities"]
    )

    dataframe["probability_percent"] = (
        dataframe["probability"] * 100
    )

    return dataframe[
        [
            "display_name",
            "probability",
            "probability_percent"
        ]
    ]


def build_feature_dataframe(features):
    if not features:
        return pd.DataFrame(
            columns=[
                "feature",
                "tfidf_value",
                "class_weight",
                "contribution"
            ]
        )

    return pd.DataFrame(features)[
        [
            "feature",
            "tfidf_value",
            "class_weight",
            "contribution"
        ]
    ]


EXAMPLE_MESSAGES = {
    "Urgent needs": (
        "Families urgently need clean water, food and medical supplies."
    ),
    "Missing person": (
        "A 14-year-old child is still missing after the floods. "
        "Please contact local authorities with any information."
    ),
    "Infrastructure damage": (
        "The earthquake destroyed several buildings and caused "
        "widespread power outages."
    ),
    "Donation effort": (
        "Volunteers are collecting food, clothing and donations "
        "for families affected by the hurricane."
    ),
    "Sympathy and support": (
        "Our thoughts and prayers are with everyone affected "
        "by this terrible disaster."
    ),
    "Custom message": ""
}


with st.sidebar:
    st.header("About CrisisText")

    st.write(
        "CrisisText classifies humanitarian crisis messages "
        "into ten operational categories."
    )

    st.metric(
        label="Final Test Accuracy",
        value="75.15%"
    )

    st.metric(
        label="Final Test Macro-F1",
        value="72.82%"
    )

    st.metric(
        label="Missing-Person Recall",
        value="73.61%"
    )

    st.metric(
        label="Urgent-Needs Recall",
        value="62.57%"
    )

    st.divider()

    st.caption(
        "The model is a TF-IDF word unigram–bigram classifier "
        "with class-balanced Logistic Regression."
    )

    st.warning(
        "Decision-support prototype only. Predictions should "
        "not replace human review during real emergencies."
    )


st.markdown(
    '<p class="main-title">🚨 CrisisText</p>',
    unsafe_allow_html=True
)

st.markdown(
    '<p class="subtitle">'
    'Explainable humanitarian crisis message triage'
    '</p>',
    unsafe_allow_html=True
)


selected_example = st.selectbox(
    "Choose an example or enter a custom message",
    options=list(EXAMPLE_MESSAGES.keys()),
    index=0
)

default_text = EXAMPLE_MESSAGES[selected_example]

message = st.text_area(
    "Crisis message",
    value=default_text,
    height=160,
    placeholder=(
        "Enter a crisis-related message, social-media post "
        "or humanitarian update..."
    )
)

top_n = st.slider(
    "Number of explanation features",
    min_value=3,
    max_value=15,
    value=8
)

analyze_button = st.button(
    "Analyze message",
    type="primary",
    use_container_width=True
)


if analyze_button:
    if not message.strip():
        st.error("Enter a message before running the analysis.")
        st.stop()

    try:
        model = get_model()

        with st.spinner("Analyzing message..."):
            result = predict_message(
                text=message,
                model=model,
                top_n=top_n
            )

        probability_df = build_probability_dataframe(result)

        confidence = result["confidence"]
        confidence_percent = confidence * 100

        st.success("Analysis completed.")

        st.markdown(
            f"""
            <div class="prediction-card">
                <div class="small-note">Predicted category</div>
                <div class="prediction-label">
                    {result["display_name"]}
                </div>
                <div class="small-note">
                    Model confidence: {confidence_percent:.2f}%
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )

        metric_column_1, metric_column_2, metric_column_3 = st.columns(3)

        with metric_column_1:
            st.metric(
                "Confidence",
                f"{confidence_percent:.2f}%"
            )

        with metric_column_2:
            st.metric(
                "Second-best category",
                probability_df.iloc[1]["display_name"]
            )

        with metric_column_3:
            margin = (
                probability_df.iloc[0]["probability"]
                - probability_df.iloc[1]["probability"]
            )

            st.metric(
                "Top-two margin",
                f"{margin:.2%}"
            )

        st.subheader("Class probabilities")

        chart_df = (
            probability_df
            .head(10)
            .set_index("display_name")[
                ["probability_percent"]
            ]
        )

        st.bar_chart(
            chart_df,
            horizontal=True
        )

        st.dataframe(
            probability_df[
                [
                    "display_name",
                    "probability_percent"
                ]
            ]
            .rename(
                columns={
                    "display_name": "Category",
                    "probability_percent": "Probability (%)"
                }
            )
            .round(2),
            use_container_width=True,
            hide_index=True
        )

        supporting_df = build_feature_dataframe(
            result["supporting_features"]
        )

        opposing_df = build_feature_dataframe(
            result["opposing_features"]
        )

        left_column, right_column = st.columns(2)

        with left_column:
            st.subheader("Supporting features")

            if supporting_df.empty:
                st.info(
                    "No positive feature contributions were found."
                )
            else:
                st.dataframe(
                    supporting_df.rename(
                        columns={
                            "feature": "Feature",
                            "tfidf_value": "TF-IDF",
                            "class_weight": "Model weight",
                            "contribution": "Contribution"
                        }
                    ).round(4),
                    use_container_width=True,
                    hide_index=True
                )

        with right_column:
            st.subheader("Opposing features")

            if opposing_df.empty:
                st.info(
                    "No opposing feature contributions were found."
                )
            else:
                st.dataframe(
                    opposing_df.rename(
                        columns={
                            "feature": "Feature",
                            "tfidf_value": "TF-IDF",
                            "class_weight": "Model weight",
                            "contribution": "Contribution"
                        }
                    ).round(4),
                    use_container_width=True,
                    hide_index=True
                )

        with st.expander("Raw prediction output"):
            st.json(
                {
                    "predicted_class": result["predicted_class"],
                    "display_name": result["display_name"],
                    "confidence": result["confidence"]
                }
            )

        st.caption(
            "Feature contribution equals the message's TF-IDF value "
            "multiplied by the selected class coefficient. "
            "It is not itself a probability."
        )

    except FileNotFoundError as error:
        st.error(str(error))

    except Exception as error:
        st.exception(error)

In [ ]:
!python -m py_compile /content/drive/MyDrive/crisis-text-triage/app.py

In [ ]:
from importlib.metadata import version
from pathlib import Path

requirements_path = Path(
    "/content/drive/MyDrive/crisis-text-triage/requirements.txt"
)

packages = [
    "streamlit",
    "pandas",
    "numpy",
    "scipy",
    "scikit-learn",
    "joblib"
]

requirements = []

for package in packages:
    requirements.append(
        f"{package}=={version(package)}"
    )

requirements_path.write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8"
)

print(requirements_path.read_text())
print("Saved:", requirements_path)

In [ ]:
!cat /content/drive/MyDrive/crisis-text-triage/requirements.txt

In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/crisis-text-triage"
)

def format_size(size_bytes):
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(size_bytes)

    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024


def print_project_tree(directory, prefix=""):
    entries = sorted(
        directory.iterdir(),
        key=lambda path: (
            path.is_file(),
            path.name.lower()
        )
    )

    for index, entry in enumerate(entries):
        is_last = index == len(entries) - 1
        branch = "└── " if is_last else "├── "

        if entry.is_dir():
            print(f"{prefix}{branch}📁 {entry.name}/")

            extension = "    " if is_last else "│   "

            print_project_tree(
                entry,
                prefix + extension
            )

        else:
            file_size = format_size(
                entry.stat().st_size
            )

            print(
                f"{prefix}{branch}📄 "
                f"{entry.name} [{file_size}]"
            )


if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_ROOT}"
    )


print("=" * 90)
print("CRISISTEXT PROJECT TREE")
print("=" * 90)
print(f"📁 {PROJECT_ROOT.name}/")

print_project_tree(PROJECT_ROOT)


all_files = [
    path
    for path in PROJECT_ROOT.rglob("*")
    if path.is_file()
]

all_directories = [
    path
    for path in PROJECT_ROOT.rglob("*")
    if path.is_dir()
]

total_size = sum(
    path.stat().st_size
    for path in all_files
)


manifest_records = []

for file_path in sorted(all_files):
    relative_path = file_path.relative_to(
        PROJECT_ROOT
    )

    file_stats = file_path.stat()

    manifest_records.append({
        "relative_path": str(relative_path),
        "file_name": file_path.name,
        "extension": file_path.suffix.lower(),
        "size_bytes": file_stats.st_size,
        "size_readable": format_size(
            file_stats.st_size
        ),
        "modified_at": datetime.fromtimestamp(
            file_stats.st_mtime
        ).isoformat(timespec="seconds")
    })


manifest_df = pd.DataFrame(
    manifest_records
)


print("\n" + "=" * 90)
print("PROJECT SUMMARY")
print("=" * 90)

print("Project root:", PROJECT_ROOT)
print("Folder count:", len(all_directories))
print("File count:", len(all_files))
print("Total size:", format_size(total_size))


expected_files = [
    "app.py",
    "requirements.txt",
    "src/inference.py",
    "models/final_e11_train_plus_validation.joblib",
    "models/selected_validation_model_e11.joblib",
    "reports/validation_results.csv",
    "reports/final_test_metrics.json",
    "reports/final_test_classification_report.csv",
    "reports/final_test_predictions.csv",
    "reports/final_test_confusion_matrix.csv",
    "reports/final_test_confusion_matrix_normalized.csv",
    "reports/final_test_confusion_matrix_normalized.png",
    "reports/final_test_confusion_pairs.csv",
    "reports/top_features_all_classes.csv",
    "reports/manual_error_audit.csv",
    "reports/manual_error_audit_summary.csv",
    "checkpoints/final_model_selection.json"
]


verification_records = []

for relative_path in expected_files:
    full_path = PROJECT_ROOT / relative_path

    verification_records.append({
        "required_file": relative_path,
        "status": (
            "READY"
            if full_path.exists()
            else "MISSING"
        ),
        "size": (
            format_size(full_path.stat().st_size)
            if full_path.exists()
            and full_path.is_file()
            else None
        )
    })


verification_df = pd.DataFrame(
    verification_records
)


print("\n" + "=" * 90)
print("IMPORTANT FILE CHECK")
print("=" * 90)

display(verification_df)


missing_files = verification_df[
    verification_df["status"] == "MISSING"
]


if missing_files.empty:
    print(
        "\n✅ All essential project files are ready."
    )
else:
    print(
        f"\n⚠️ Missing essential files: "
        f"{len(missing_files)}"
    )

    display(missing_files)


manifest_path = (
    PROJECT_ROOT
    / "project_manifest.csv"
)

manifest_df.to_csv(
    manifest_path,
    index=False
)


tree_output_path = (
    PROJECT_ROOT
    / "project_tree.txt"
)


tree_lines = []

def collect_tree(directory, prefix=""):
    entries = sorted(
        directory.iterdir(),
        key=lambda path: (
            path.is_file(),
            path.name.lower()
        )
    )

    for index, entry in enumerate(entries):
        is_last = index == len(entries) - 1
        branch = "└── " if is_last else "├── "

        if entry.is_dir():
            tree_lines.append(
                f"{prefix}{branch}{entry.name}/"
            )

            extension = (
                "    "
                if is_last
                else "│   "
            )

            collect_tree(
                entry,
                prefix + extension
            )

        else:
            tree_lines.append(
                f"{prefix}{branch}"
                f"{entry.name} "
                f"[{format_size(entry.stat().st_size)}]"
            )


tree_lines.append(
    f"{PROJECT_ROOT.name}/"
)

collect_tree(PROJECT_ROOT)

tree_output_path.write_text(
    "\n".join(tree_lines),
    encoding="utf-8"
)


print("\nManifest saved:")
print(manifest_path)

print("\nTree saved:")
print(tree_output_path)